# WiFi Module2 V7.1 Ablation

Source-only soft tri-state router ablation with six variants:
- v71_base (original V7_C)
- v71_classwise
- v71_atten
- v71_synthopt
- v71_rank
- v71_hardneg

This notebook does **not** use target-domain state labels for router calibration.

In [ ]:
import os, json, time, itertools
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# ---- subset sizes ----
TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4
TX_SPLIT_REPEATS = 15

# ---- protocol libraries ----
RX_PROTOCOL_LIBRARY = {
    "3-9": dict(source_rx_num=3, drift_rx_num=9),
    "6-6": dict(source_rx_num=6, drift_rx_num=6),
    "9-3": dict(source_rx_num=9, drift_rx_num=3),
}
TX_PROTOCOL_LIBRARY = {
    "2-4": dict(known_tx_num=2, unknown_tx_num=4),
    "3-3": dict(known_tx_num=3, unknown_tx_num=3),
    "4-2": dict(known_tx_num=4, unknown_tx_num=2),
}

# 4 representative scenarios used in the recent analysis thread.
SELECTED_PROTOCOL_COMBOS = [
    ("9-3", "4-2"),
    ("9-3", "2-4"),
    ("6-6", "3-3"),
    ("3-9", "2-4"),
]

# ---- protocol ----
TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

# ---- signals ----
Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

# ---- training ----
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3

# ---- dims ----
D_DIM = 32

# ---- Sid fusion ----
FUSION_LAM = 0.5

# ---- calibration targets ----
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05

# ---- source-calibrated router ----
ROUTER_CAL_FRAC = 0.20
ROUTER_MIN_CAL_PER_POOL = 256
ROUTER_TOP2_TEMP = 0.20
DOM_STATS_SID_Q = 0.60          # keep samples above the per-class Sid quantile
DOM_STATS_MIN_KEEP_PER_CLASS = 64
ROUTER_FEATURE_MODE = "hardk"   # {"hardk", "top2soft"}

# NOTE:
# v6 calibrates the tri-state router only on stable source holdout A.
# B/C/D/E/F are used only as unlabeled evaluation pools.


# ---- v7.1 ablation registry ----
V71_VARIANT_ORDER = [
    "v71_base",
    "v71_classwise",
    "v71_atten",
    "v71_synthopt",
    "v71_rank",
    "v71_hardneg",
]

# All variants remain source-only and keep the soft tri-state router.
V71_VARIANT_SPECS = {
    # Original V7_C: global synthetic-drift calibrator + soft tri-state router.
    "v71_base": dict(
        classwise=False,
        atten=False,
        synthopt=False,
        rank=False,
        hardneg=False,
    ),
    # Suggestion 1: class-conditional drift calibrator while keeping the soft router.
    "v71_classwise": dict(
        classwise=True,
        atten=False,
        synthopt=False,
        rank=False,
        hardneg=False,
    ),
    # Suggestion 2: stable-protection attenuation on top of the original V7_C.
    "v71_atten": dict(
        classwise=False,
        atten=True,
        synthopt=False,
        rank=False,
        hardneg=False,
    ),
    # Suggestion 3: improved source-only synthetic drift generation.
    "v71_synthopt": dict(
        classwise=False,
        atten=False,
        synthopt=True,
        rank=False,
        hardneg=False,
    ),
    # Suggestion 4: ranking-style calibration objective on source-only stable vs synthetic drift scores.
    "v71_rank": dict(
        classwise=False,
        atten=False,
        synthopt=False,
        rank=True,
        hardneg=False,
    ),
    # Suggestion 5: add source-only hard negative stable tails to protect stable samples.
    "v71_hardneg": dict(
        classwise=False,
        atten=False,
        synthopt=False,
        rank=False,
        hardneg=True,
    ),
}

# ---- synthetic drift calibrator (source-only) ----
SYNTH_DRIFT_COPIES = 1
SYNTH_SHIFT_MAX = 4
SYNTH_PHASE_SLOPE_MAX = 0.08
SYNTH_GAIN_MIN = 0.85
SYNTH_GAIN_MAX = 1.15
SYNTH_IQ_LEAK_MAX = 0.08
SYNTH_NOISE_STD_MIN = 0.005
SYNTH_NOISE_STD_MAX = 0.03
SYNTH_MIN_CLASS_CAL = 24

# Improved synthetic drift generator (source-only; still no target usage)
SYNTHOPT_DRIFT_COPIES = 2
SYNTHOPT_SHIFT_MAX = 3
SYNTHOPT_PHASE_SLOPE_MAX = 0.05
SYNTHOPT_GAIN_MIN = 0.90
SYNTHOPT_GAIN_MAX = 1.10
SYNTHOPT_IQ_LEAK_MAX = 0.05
SYNTHOPT_NOISE_STD_MIN = 0.003
SYNTHOPT_NOISE_STD_MAX = 0.020
SYNTHOPT_FREQ_TILT_MAX = 0.18
SYNTHOPT_COLOR_NOISE_STD_MIN = 0.002
SYNTHOPT_COLOR_NOISE_STD_MAX = 0.012
SYNTHOPT_TIME_WARP_MAX = 0.03

# Stable-protection attenuation (source-only calibrated)
ATTEN_ALPHA = 0.45
ATTEN_SID_Q = 0.75
ATTEN_DRIFTREF_Q = 0.60

# Ranking-style calibrator
RANK_MAX_PAIRS = 50000

# Hard-negative stable tail mining
HARDNEG_Q = 0.85
HARDNEG_DUP = 3

# ---- outputs ----
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"../owdc_results/results_wisig_module2_v7_1/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED,
    TX_TOTAL_USE=TX_TOTAL_USE,
    RX_TOTAL_USE=RX_TOTAL_USE,
    DAY_TOTAL_USE=DAY_TOTAL_USE,
    TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    RX_PROTOCOL_LIBRARY=RX_PROTOCOL_LIBRARY,
    TX_PROTOCOL_LIBRARY=TX_PROTOCOL_LIBRARY,
    SELECTED_PROTOCOL_COMBOS=SELECTED_PROTOCOL_COMBOS,
    TRAIN_DATES=TRAIN_DATES,
    TEST_DATES_RX=TEST_DATES_RX,
    TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ,
    D_FROM=D_FROM,
    MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES,
    DROPOUT=DROPOUT,
    D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM,
    FRR_TARGET=FRR_TARGET,
    FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
    ROUTER_CAL_FRAC=ROUTER_CAL_FRAC,
    ROUTER_MIN_CAL_PER_POOL=ROUTER_MIN_CAL_PER_POOL,
    ROUTER_TOP2_TEMP=ROUTER_TOP2_TEMP,
    ROUTER_FEATURE_MODE=ROUTER_FEATURE_MODE,
    DOM_STATS_SID_Q=DOM_STATS_SID_Q,
    DOM_STATS_MIN_KEEP_PER_CLASS=DOM_STATS_MIN_KEEP_PER_CLASS,
    V71_VARIANT_ORDER=V71_VARIANT_ORDER,
    V71_VARIANT_SPECS=V71_VARIANT_SPECS,
    SYNTH_DRIFT_COPIES=SYNTH_DRIFT_COPIES,
    SYNTH_SHIFT_MAX=SYNTH_SHIFT_MAX,
    SYNTH_PHASE_SLOPE_MAX=SYNTH_PHASE_SLOPE_MAX,
    SYNTH_GAIN_MIN=SYNTH_GAIN_MIN,
    SYNTH_GAIN_MAX=SYNTH_GAIN_MAX,
    SYNTH_IQ_LEAK_MAX=SYNTH_IQ_LEAK_MAX,
    SYNTH_NOISE_STD_MIN=SYNTH_NOISE_STD_MIN,
    SYNTH_NOISE_STD_MAX=SYNTH_NOISE_STD_MAX,
    SYNTH_MIN_CLASS_CAL=SYNTH_MIN_CLASS_CAL,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("RUN_DIR:", RUN_DIR)




# =========================
# Data utilities
# =========================
def get_idx(lst, val): 
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

# =========================
# Protocol suite construction
# =========================
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)

def build_unique_tx_splits(tx_use, known_tx_num, n_splits, seed=42):
    all_known = [tuple(c) for c in itertools.combinations(list(tx_use), known_tx_num)]
    if n_splits > len(all_known):
        raise ValueError(
            f"Requested TX_SPLIT_REPEATS={n_splits}, but only {len(all_known)} unique TX splits "
            f"exist for C({len(tx_use)},{known_tx_num})."
        )
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(all_known))
    tx_splits = []
    for local_id, idx in enumerate(order[:n_splits], start=1):
        known = list(all_known[idx])
        unknown = [tx for tx in tx_use if tx not in known]
        tx_splits.append({
            "split_id": local_id,
            "known": known,
            "unknown": unknown,
        })
    return tx_splits

def build_protocol_specs(
    tx_use, rx_use,
    protocol_combos,
    rx_protocol_library, tx_protocol_library,
    tx_split_repeats, seed=42
):
    protocol_specs = []
    for combo_idx, (rx_tag, tx_tag) in enumerate(protocol_combos, start=1):
        if rx_tag not in rx_protocol_library:
            raise KeyError(f"Unknown RX protocol tag: {rx_tag}")
        if tx_tag not in tx_protocol_library:
            raise KeyError(f"Unknown TX protocol tag: {tx_tag}")

        rx_cfg = dict(rx_protocol_library[rx_tag])
        tx_cfg = dict(tx_protocol_library[tx_tag])

        src_n = int(rx_cfg["source_rx_num"])
        drift_n = int(rx_cfg["drift_rx_num"])
        known_n = int(tx_cfg["known_tx_num"])
        unknown_n = int(tx_cfg["unknown_tx_num"])

        if src_n + drift_n != len(rx_use):
            raise ValueError(
                f"RX protocol {rx_tag} expects source+drift={src_n + drift_n}, "
                f"but len(RX_USE)={len(rx_use)}."
            )
        if known_n + unknown_n != len(tx_use):
            raise ValueError(
                f"TX protocol {tx_tag} expects known+unknown={known_n + unknown_n}, "
                f"but len(TX_USE)={len(tx_use)}."
            )

        rng_rx = np.random.RandomState(seed + 1000 * combo_idx + 17 * src_n + drift_n)
        source_rxs = list(rng_rx.choice(list(rx_use), size=src_n, replace=False))
        source_rxs.sort()
        drift_rx = [r for r in rx_use if r not in source_rxs]

        tx_splits = build_unique_tx_splits(
            tx_use=tx_use,
            known_tx_num=known_n,
            n_splits=tx_split_repeats,
            seed=seed + 5000 + 97 * combo_idx,
        )

        protocol_specs.append(dict(
            protocol_tag=f"RX{rx_tag}_TX{tx_tag}",
            rx_protocol=rx_tag,
            tx_protocol=tx_tag,
            source_rx_num=src_n,
            drift_rx_num=drift_n,
            known_tx_num=known_n,
            unknown_tx_num=unknown_n,
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            tx_splits=tx_splits,
        ))
    return protocol_specs

PROTOCOL_SPECS = build_protocol_specs(
    TX_USE, RX_USE,
    SELECTED_PROTOCOL_COMBOS,
    RX_PROTOCOL_LIBRARY, TX_PROTOCOL_LIBRARY,
    tx_split_repeats=TX_SPLIT_REPEATS,
    seed=SEED + 777,
)

with open(os.path.join(RUN_DIR, "protocol_specs.json"), "w", encoding="utf-8") as f:
    json.dump(PROTOCOL_SPECS, f, indent=2)

print("Selected protocol combinations:", len(PROTOCOL_SPECS))
for spec in PROTOCOL_SPECS:
    print(
        f"[{spec['protocol_tag']}] "
        f"SOURCE_RXS={spec['source_rxs']} | DRIFT_RX={spec['drift_rx']} | "
        f"TXsplits={len(spec['tx_splits'])}"
    )


# =========================
# Model: ResNet18-1D
# =========================
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

# =========================
# Train / inference helpers
# =========================
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): 
        return self.X.shape[0]
    def __getitem__(self, i): 
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def roc_auc(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

def l2norm(x, axis=1, eps=1e-12):
    return x/(np.linalg.norm(x,axis=axis,keepdims=True)+eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        P[k] = ZN[y==k].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    return l2norm(Z,axis=1) @ P.T

def sid_EmbMaha(Z, mu_z, var_z):
    dif = Z[:,None,:] - mu_z[None,:,:]
    dist = np.sum((dif*dif)/(var_z[None,:,:] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_Energy(logits):
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits-m).sum(axis=1, keepdims=True)+1e-12)).squeeze(1).astype(np.float32)

def zscore(x, ref):
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if std < 1e-12:
        std = 1.0
    return ((x - mu) / std).astype(np.float32)

# =========================
# Domain (Sdom): V1_mixNLL
# =========================
def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps=[]
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum() + 1e-12) - np.log(R)
        out[i] = float(-loglik)  # higher => more drift
    return out

# =========================
# Build datasets
# =========================
def build_source_train(compact_dataset, KNOWN_TX, source_rxs):
    X_list, y_list, D_list = [], [], []
    rx_id_list = []
    for tx in KNOWN_TX:
        for rx in source_rxs:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    X = np.concatenate(X_list,0).astype(np.float32)
    y = np.concatenate(y_list,0).astype(np.int64)
    D = np.concatenate(D_list,0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list,0).astype(np.int64)
    return X,y,D,rx_id

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, D_list = [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); D_list.append(Df)
    X = np.concatenate(X_list,0).astype(np.float32)
    D = np.concatenate(D_list,0).astype(np.float32)
    return X, D

# =========================
# Source-calibrated router + plots
# =========================
def fit_classwise_dom_stats(Sdom_train, y_train, K, min_std=1e-3):
    Sdom_train = np.asarray(Sdom_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int64)

    global_mu = float(np.mean(Sdom_train))
    global_std = float(max(np.std(Sdom_train), min_std))

    dom_mu = np.full((K,), global_mu, dtype=np.float32)
    dom_std = np.full((K,), global_std, dtype=np.float32)

    for k in range(K):
        vals = Sdom_train[y_train == k]
        if vals.size >= 8:
            dom_mu[k] = float(np.mean(vals))
            dom_std[k] = float(max(np.std(vals), min_std))
    return dom_mu, dom_std, global_mu, global_std


def select_high_sid_subset_by_class(Sid, y, K, q=0.60, min_keep_per_class=64):
    Sid = np.asarray(Sid, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)

    keep = np.zeros(len(y), dtype=bool)
    cls_thr = np.zeros((K,), dtype=np.float32)
    cls_counts = np.zeros((K,), dtype=np.int64)

    for k in range(K):
        idx = np.where(y == k)[0]
        if idx.size == 0:
            continue
        vals = Sid[idx]
        if idx.size <= max(2 * int(min_keep_per_class), 16):
            keep[idx] = True
            cls_thr[k] = float(np.min(vals))
            cls_counts[k] = int(idx.size)
            continue

        thr = float(np.quantile(vals, q))
        mask_k = vals >= thr
        if int(mask_k.sum()) < int(min_keep_per_class):
            order = np.argsort(vals)[::-1]
            take = min(int(min_keep_per_class), idx.size)
            mask_k = np.zeros(idx.size, dtype=bool)
            mask_k[order[:take]] = True
            thr = float(vals[order[take - 1]])
        keep[idx[mask_k]] = True
        cls_thr[k] = thr
        cls_counts[k] = int(mask_k.sum())

    return keep, cls_thr, cls_counts

def normalize_dom_matrix_by_class(Sdom_allk, dom_mu, dom_std, global_mu, global_std):
    Sdom_allk = np.asarray(Sdom_allk, dtype=np.float32)
    K = Sdom_allk.shape[1]
    out = np.zeros_like(Sdom_allk, dtype=np.float32)
    for k in range(K):
        mu = float(dom_mu[k]) if k < len(dom_mu) else float(global_mu)
        sd = float(dom_std[k]) if k < len(dom_std) else float(global_std)
        out[:, k] = (Sdom_allk[:, k] - mu) / max(sd, 1e-6)
    return out.astype(np.float32)

def sdom_V1_mixNLL_allk(D_eval, K, models_kr, fallback_k, source_rx_ids):
    D_eval = np.asarray(D_eval, dtype=np.float32)
    N = D_eval.shape[0]
    R = len(source_rx_ids)
    out = np.zeros((N, K), dtype=np.float32)
    for k in range(K):
        logps = []
        for rxid in source_rx_ids:
            mu, Sinv, logdet = models_kr.get((k, rxid), fallback_k[k])
            logps.append(logpdf_gaussian(D_eval, mu, Sinv, logdet).astype(np.float32))
        logps = np.stack(logps, axis=1)  # [N, R]
        m = np.max(logps, axis=1, keepdims=True)
        loglik = m[:, 0] + np.log(np.exp(logps - m).sum(axis=1) + 1e-12) - np.log(R)
        out[:, k] = (-loglik).astype(np.float32)
    return out

def gather_hard_class_scores(score_allk, k_hat):
    score_allk = np.asarray(score_allk, dtype=np.float32)
    k_hat = np.asarray(k_hat, dtype=np.int64)
    return score_allk[np.arange(len(k_hat)), k_hat].astype(np.float32)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float32)
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.clip(np.sum(e, axis=axis, keepdims=True), 1e-12, None)

def aggregate_top2_scores(score_allk, id_score_allk, temp=0.20):
    score_allk = np.asarray(score_allk, dtype=np.float32)
    id_score_allk = np.asarray(id_score_allk, dtype=np.float32)

    top2_idx = np.argsort(id_score_allk, axis=1)[:, -2:][:, ::-1]
    top2_id_scores = np.take_along_axis(id_score_allk, top2_idx, axis=1)
    weights = softmax_np(top2_id_scores / max(float(temp), 1e-6), axis=1)
    top2_scores = np.take_along_axis(score_allk, top2_idx, axis=1)
    agg = np.sum(weights * top2_scores, axis=1)
    return agg.astype(np.float32), top2_idx.astype(np.int64), weights.astype(np.float32)

def build_router_features(Sid, Sdom_raw, Sdrift_norm):
    return np.stack(
        [
            np.asarray(Sid, dtype=np.float32),
            np.asarray(Sdom_raw, dtype=np.float32),
            np.asarray(Sdrift_norm, dtype=np.float32),
        ],
        axis=1,
    ).astype(np.float32)

def split_cal_eval_indices(n, frac, rng, min_cal=256):
    if n < 4:
        idx = np.arange(n)
        cut = max(1, n // 2)
        return idx[:cut], idx[cut:]
    n_cal = max(1, int(round(frac * n)))
    if n >= 2 * min_cal:
        n_cal = max(n_cal, min_cal)
    n_cal = min(n_cal, n - 1)
    perm = rng.permutation(n)
    return perm[:n_cal], perm[n_cal:]



def sigmoid_np(x):
    x = np.asarray(x, dtype=np.float32)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40.0, 40.0)))


def robust_scale(vals, min_scale=1e-3):
    vals = np.asarray(vals, dtype=np.float32)
    sd = float(np.std(vals))
    q75, q25 = np.quantile(vals, [0.75, 0.25])
    iqr_sd = float((q75 - q25) / 1.349) if q75 > q25 else 0.0
    return max(sd, iqr_sd, float(min_scale))


def calibrate_source_soft_router(Sid_stable, Sdrift_stable, frr_target=0.05, false_drift_target=0.05):
    Sid_stable = np.asarray(Sid_stable, dtype=np.float32)
    Sdrift_stable = np.asarray(Sdrift_stable, dtype=np.float32)

    tau_id = float(np.quantile(Sid_stable, frr_target))
    tau_drift = float(np.quantile(Sdrift_stable, 1.0 - false_drift_target))
    temp_id = float(robust_scale(Sid_stable))
    temp_drift = float(robust_scale(Sdrift_stable))

    return dict(tau_id=tau_id, tau_drift=tau_drift, temp_id=temp_id, temp_drift=temp_drift)


def predict_source_soft_router(Sid, Sdrift_norm, router_cfg):
    Sid = np.asarray(Sid, dtype=np.float32)
    Sdrift_norm = np.asarray(Sdrift_norm, dtype=np.float32)

    tau_id = float(router_cfg["tau_id"])
    tau_drift = float(router_cfg["tau_drift"])
    temp_id = max(float(router_cfg["temp_id"]), 1e-6)
    temp_drift = max(float(router_cfg["temp_drift"]), 1e-6)

    p_unknown = sigmoid_np((tau_id - Sid) / temp_id)
    p_known = 1.0 - p_unknown
    p_drift_cond = sigmoid_np((Sdrift_norm - tau_drift) / temp_drift)

    p_stable = p_known * (1.0 - p_drift_cond)
    p_drift = p_known * p_drift_cond

    probs = np.stack([p_stable, p_drift, p_unknown], axis=1).astype(np.float32)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)
    pred = np.argmax(probs, axis=1).astype(np.int64)
    return pred, probs


def calibrate_thresholds(Sid_stable, Sdom_stable, frr_target=0.05, fdr_target=0.05):
    tau_id = float(np.quantile(Sid_stable, frr_target))
    tau_dom = float(np.quantile(Sdom_stable, 1.0 - fdr_target))
    return tau_id, tau_dom

def gate_predict(Sid, Sdom, tau_id, tau_dom):
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
    return pred

def plot_gate_scatter(Sid, Sdom, gt, tau_id, tau_dom, out_png, max_points=20000):
    n = len(Sid)
    if n > max_points:
        idx = np.random.RandomState(SEED).choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6,5))
    plt.scatter(Sid[idx], Sdom[idx], s=3, c=gt[idx], alpha=0.5)
    plt.axvline(tau_id, linestyle="--")
    plt.axhline(tau_dom, linestyle="--")
    plt.xlabel("S_id")
    plt.ylabel("S_dom_raw")
    plt.title("Baseline hard gate (held-out eval)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def plot_joint_scatter(Sid, Sdrift, gt, out_png, max_points=20000):
    n = len(Sid)
    if n > max_points:
        idx = np.random.RandomState(SEED).choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6,5))
    plt.scatter(Sid[idx], Sdrift[idx], s=3, c=gt[idx], alpha=0.5)
    plt.xlabel("S_id")
    plt.ylabel("S_drift_norm")
    plt.title("Source-calibrated v6 features (eval)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def plot_confmat(cm, out_png, title):
    plt.figure(figsize=(5,4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xticks([0,1,2], ["Stable","Drift","Unknown"])
    plt.yticks([0,1,2], ["Stable","Drift","Unknown"])
    plt.xlabel("Pred")
    plt.ylabel("GT")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def compute_state_metrics(pred_A, pred_B, pred_C, pred_D, pred_E, pred_F):
    gt = np.concatenate([
        np.zeros_like(pred_A, dtype=np.int64),
        np.ones_like(pred_B, dtype=np.int64),
        np.ones_like(pred_C, dtype=np.int64),
        np.full_like(pred_D, 2, dtype=np.int64),
        np.full_like(pred_E, 2, dtype=np.int64),
        np.full_like(pred_F, 2, dtype=np.int64),
    ])
    pred = np.concatenate([pred_A, pred_B, pred_C, pred_D, pred_E, pred_F])
    cm = confusion_matrix(gt, pred, labels=[0,1,2])

    stable_acc_gate = float((pred_A == 0).mean())
    drift_acc_rx = float((pred_B == 1).mean())
    drift_acc_day = float((pred_C == 1).mean())
    drift_preds = np.concatenate([pred_B, pred_C])
    unknown_preds = np.concatenate([pred_D, pred_E, pred_F])

    drift_acc_all = float((drift_preds == 1).mean())
    unknown_reject_in = float((pred_D == 2).mean())
    unknown_reject_rx = float((pred_E == 2).mean())
    unknown_reject_day = float((pred_F == 2).mean())
    unknown_reject_all = float((unknown_preds == 2).mean())

    FRR_stable = float((pred_A != 0).mean())
    false_drift_stable = float((pred_A == 1).mean())
    false_unknown_stable = float((pred_A == 2).mean())

    FAR_unknown_accept = float((unknown_preds != 2).mean())
    FAR_unknown_accept_in = float((pred_D != 2).mean())
    FAR_unknown_accept_rx = float((pred_E != 2).mean())
    FAR_unknown_accept_day = float((pred_F != 2).mean())

    false_drift_unknown = float((unknown_preds == 1).mean())
    false_drift_unknown_in = float((pred_D == 1).mean())
    false_drift_unknown_rx = float((pred_E == 1).mean())
    false_drift_unknown_day = float((pred_F == 1).mean())

    false_stable_unknown = float((unknown_preds == 0).mean())
    false_stable_unknown_in = float((pred_D == 0).mean())
    false_stable_unknown_rx = float((pred_E == 0).mean())
    false_stable_unknown_day = float((pred_F == 0).mean())

    miss_drift_rx = float((pred_B == 0).mean())
    miss_drift_day = float((pred_C == 0).mean())
    miss_drift_all = float((np.sum(pred_B == 0) + np.sum(pred_C == 0)) / (len(pred_B) + len(pred_C)))

    false_unknown_drift_rx = float((pred_B == 2).mean())
    false_unknown_drift_day = float((pred_C == 2).mean())
    false_unknown_drift_all = float((np.sum(pred_B == 2) + np.sum(pred_C == 2)) / (len(pred_B) + len(pred_C)))

    metrics = dict(
        stable_acc_gate=stable_acc_gate,
        drift_acc_rx=drift_acc_rx,
        drift_acc_day=drift_acc_day,
        drift_acc_all=drift_acc_all,
        unknown_reject_in=unknown_reject_in,
        unknown_reject_rx=unknown_reject_rx,
        unknown_reject_day=unknown_reject_day,
        unknown_reject_all=unknown_reject_all,
        FRR_stable=FRR_stable,
        false_drift_stable=false_drift_stable,
        false_unknown_stable=false_unknown_stable,
        FAR_unknown_accept=FAR_unknown_accept,
        FAR_unknown_accept_in=FAR_unknown_accept_in,
        FAR_unknown_accept_rx=FAR_unknown_accept_rx,
        FAR_unknown_accept_day=FAR_unknown_accept_day,
        false_drift_unknown=false_drift_unknown,
        false_drift_unknown_in=false_drift_unknown_in,
        false_drift_unknown_rx=false_drift_unknown_rx,
        false_drift_unknown_day=false_drift_unknown_day,
        false_stable_unknown=false_stable_unknown,
        false_stable_unknown_in=false_stable_unknown_in,
        false_stable_unknown_rx=false_stable_unknown_rx,
        false_stable_unknown_day=false_stable_unknown_day,
        miss_drift_rx=miss_drift_rx,
        miss_drift_day=miss_drift_day,
        miss_drift_all=miss_drift_all,
        false_unknown_drift_rx=false_unknown_drift_rx,
        false_unknown_drift_day=false_unknown_drift_day,
        false_unknown_drift_all=false_unknown_drift_all,
    )
    return metrics, cm, gt, pred

def prefix_keys(d, prefix):
    return {f"{prefix}{k}": v for k, v in d.items()}



# =========================
# V7.1 ablation helpers (source-only; no target-state calibration)
# =========================
def complex_to_iq(x_c):
    return np.stack([np.real(x_c), np.imag(x_c)], axis=-1).astype(np.float32)

def synthetic_drift_augment_batch(X_256_2, rng, copies=1):
    X = np.asarray(X_256_2, dtype=np.float32)
    if X.ndim != 3 or X.shape[-1] != 2:
        raise ValueError(f"Expected [N, L, 2], got {X.shape}")
    Xc = iq_to_complex(X)
    N, L = Xc.shape
    n = np.arange(L, dtype=np.float32).reshape(1, L)

    outs = []
    for _ in range(int(max(1, copies))):
        xc = Xc.copy()

        shifts = rng.randint(-SYNTH_SHIFT_MAX, SYNTH_SHIFT_MAX + 1, size=N)
        for i, s in enumerate(shifts):
            if s != 0:
                xc[i] = np.roll(xc[i], int(s))

        phase0 = rng.uniform(-np.pi, np.pi, size=(N, 1)).astype(np.float32)
        slope = rng.uniform(-SYNTH_PHASE_SLOPE_MAX, SYNTH_PHASE_SLOPE_MAX, size=(N, 1)).astype(np.float32)
        xc = xc * np.exp(1j * (phase0 + slope * n)).astype(np.complex64)

        gain = rng.uniform(SYNTH_GAIN_MIN, SYNTH_GAIN_MAX, size=(N, 1)).astype(np.float32)
        xc = xc * gain

        leak = rng.uniform(-SYNTH_IQ_LEAK_MAX, SYNTH_IQ_LEAK_MAX, size=(N, 1)).astype(np.float32)
        i_gain = rng.uniform(0.95, 1.05, size=(N, 1)).astype(np.float32)
        q_gain = rng.uniform(0.95, 1.05, size=(N, 1)).astype(np.float32)
        I = np.real(xc).astype(np.float32)
        Q = np.imag(xc).astype(np.float32)
        I2 = i_gain * I + leak * Q
        Q2 = q_gain * Q + leak * I
        xc = (I2 + 1j * Q2).astype(np.complex64)

        noise_std = rng.uniform(SYNTH_NOISE_STD_MIN, SYNTH_NOISE_STD_MAX, size=(N, 1)).astype(np.float32)
        noise = (rng.randn(N, L) + 1j * rng.randn(N, L)).astype(np.complex64)
        xc = xc + noise_std * noise

        rms = np.sqrt(np.mean(np.abs(xc) ** 2, axis=1, keepdims=True) + 1e-12).astype(np.float32)
        xc = (xc / rms).astype(np.complex64)

        outs.append(complex_to_iq(xc))

    return np.concatenate(outs, axis=0).astype(np.float32)


def _smooth_1d_reflect(x, kernel):
    pad = len(kernel) // 2
    xpad = np.pad(x, (pad, pad), mode="reflect")
    return np.convolve(xpad, kernel, mode="valid")


def synthetic_drift_augment_batch_v2(X_256_2, rng, copies=1):
    """Improved source-only synthetic drift generator.
    Adds milder time shift, frequency tilt, colored noise and slight resampling-style warping.
    Still uses only source-domain samples and labels.
    """
    X = np.asarray(X_256_2, dtype=np.float32)
    if X.ndim != 3 or X.shape[-1] != 2:
        raise ValueError(f"Expected [N, L, 2], got {X.shape}")
    Xc = iq_to_complex(X)
    N, L = Xc.shape
    n = np.arange(L, dtype=np.float32)
    outs = []
    for _ in range(int(max(1, copies))):
        xc = Xc.copy()

        shifts = rng.randint(-SYNTHOPT_SHIFT_MAX, SYNTHOPT_SHIFT_MAX + 1, size=N)
        for i, s in enumerate(shifts):
            if s != 0:
                xc[i] = np.roll(xc[i], int(s))

        # light global phase + linear phase slope
        phase0 = rng.uniform(-np.pi / 2, np.pi / 2, size=(N, 1)).astype(np.float32)
        slope = rng.uniform(-SYNTHOPT_PHASE_SLOPE_MAX, SYNTHOPT_PHASE_SLOPE_MAX, size=(N, 1)).astype(np.float32)
        xc = xc * np.exp(1j * (phase0 + slope * n.reshape(1, L))).astype(np.complex64)

        # mild time warp by linear interpolation on I/Q separately
        for i in range(N):
            warp = float(rng.uniform(-SYNTHOPT_TIME_WARP_MAX, SYNTHOPT_TIME_WARP_MAX))
            if abs(warp) > 1e-6:
                grid = np.linspace(0, L - 1, L, dtype=np.float32)
                src_grid = np.clip((1.0 + warp) * (grid - (L - 1) / 2.0) + (L - 1) / 2.0, 0, L - 1)
                re = np.interp(grid, src_grid, np.real(xc[i]).astype(np.float32))
                im = np.interp(grid, src_grid, np.imag(xc[i]).astype(np.float32))
                xc[i] = (re + 1j * im).astype(np.complex64)

        # gentle gain
        gain = rng.uniform(SYNTHOPT_GAIN_MIN, SYNTHOPT_GAIN_MAX, size=(N, 1)).astype(np.float32)
        xc = xc * gain

        # mild IQ imbalance
        leak = rng.uniform(-SYNTHOPT_IQ_LEAK_MAX, SYNTHOPT_IQ_LEAK_MAX, size=(N, 1)).astype(np.float32)
        i_gain = rng.uniform(0.97, 1.03, size=(N, 1)).astype(np.float32)
        q_gain = rng.uniform(0.97, 1.03, size=(N, 1)).astype(np.float32)
        I = np.real(xc).astype(np.float32)
        Q = np.imag(xc).astype(np.float32)
        I2 = i_gain * I + leak * Q
        Q2 = q_gain * Q + leak * I
        xc = (I2 + 1j * Q2).astype(np.complex64)

        # mild frequency tilt / front-end coloration
        f = np.linspace(-1.0, 1.0, L, dtype=np.float32)
        for i in range(N):
            tilt = float(rng.uniform(-SYNTHOPT_FREQ_TILT_MAX, SYNTHOPT_FREQ_TILT_MAX))
            Xf = np.fft.fft(xc[i])
            amp = np.clip(1.0 + tilt * f, 0.6, 1.4).astype(np.float32)
            xc[i] = np.fft.ifft(Xf * amp).astype(np.complex64)

        # white noise
        noise_std = rng.uniform(SYNTHOPT_NOISE_STD_MIN, SYNTHOPT_NOISE_STD_MAX, size=(N, 1)).astype(np.float32)
        noise = (rng.randn(N, L) + 1j * rng.randn(N, L)).astype(np.complex64)
        xc = xc + noise_std * noise

        # colored noise (low-pass filtered)
        kernel = np.array([1.0, 2.0, 3.0, 2.0, 1.0], dtype=np.float32)
        kernel = kernel / np.sum(kernel)
        color_std = rng.uniform(SYNTHOPT_COLOR_NOISE_STD_MIN, SYNTHOPT_COLOR_NOISE_STD_MAX, size=(N, 1)).astype(np.float32)
        for i in range(N):
            nr = _smooth_1d_reflect(rng.randn(L).astype(np.float32), kernel)
            ni = _smooth_1d_reflect(rng.randn(L).astype(np.float32), kernel)
            xc[i] = xc[i] + color_std[i, 0] * (nr + 1j * ni).astype(np.complex64)

        rms = np.sqrt(np.mean(np.abs(xc) ** 2, axis=1, keepdims=True) + 1e-12).astype(np.float32)
        xc = (xc / rms).astype(np.complex64)
        outs.append(complex_to_iq(xc))
    return np.concatenate(outs, axis=0).astype(np.float32)

def calibrate_classwise_soft_router(Sdrift_stable_true, y_true, K, false_drift_target=0.05, min_count=24):
    Sdrift_stable_true = np.asarray(Sdrift_stable_true, dtype=np.float32)
    y_true = np.asarray(y_true, dtype=np.int64)

    global_tau = float(np.quantile(Sdrift_stable_true, 1.0 - false_drift_target))
    global_temp = float(robust_scale(Sdrift_stable_true))

    tau = np.full((K,), global_tau, dtype=np.float32)
    temp = np.full((K,), global_temp, dtype=np.float32)

    for k in range(K):
        vals = Sdrift_stable_true[y_true == k]
        if vals.size >= max(int(min_count), 8):
            tau[k] = float(np.quantile(vals, 1.0 - false_drift_target))
            temp[k] = float(robust_scale(vals))
    return dict(
        tau_drift_by_class=tau,
        temp_drift_by_class=temp,
        tau_drift_global=global_tau,
        temp_drift_global=global_temp,
    )

def fit_binary_score_calibrator(neg_scores, pos_scores):
    neg_scores = np.asarray(neg_scores, dtype=np.float32).reshape(-1, 1)
    pos_scores = np.asarray(pos_scores, dtype=np.float32).reshape(-1, 1)

    if neg_scores.shape[0] < 4 or pos_scores.shape[0] < 4:
        thr = float(np.quantile(neg_scores[:, 0], 0.95)) if neg_scores.shape[0] > 0 else 0.0
        return dict(kind="threshold", tau=thr, temp=max(float(np.std(neg_scores[:, 0])), 1e-3))

    X = np.concatenate([neg_scores, pos_scores], axis=0)
    y = np.concatenate([
        np.zeros((neg_scores.shape[0],), dtype=np.int64),
        np.ones((pos_scores.shape[0],), dtype=np.int64),
    ])
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs")
    clf.fit(Xs, y)
    return dict(kind="logreg", scaler=scaler, clf=clf)


def fit_pairwise_rank_calibrator(neg_scores, pos_scores, max_pairs=RANK_MAX_PAIRS, seed=SEED):
    """Pairwise ranking surrogate using source-only stable vs synthetic drift scores.
    Learns a scale from pairwise score differences, then maps raw scores with a sigmoid around a source-only threshold.
    """
    neg_scores = np.asarray(neg_scores, dtype=np.float32).reshape(-1)
    pos_scores = np.asarray(pos_scores, dtype=np.float32).reshape(-1)
    if neg_scores.size < 4 or pos_scores.size < 4:
        return fit_binary_score_calibrator(neg_scores, pos_scores)
    rng = np.random.RandomState(seed + 17)
    n_pairs = int(min(max_pairs, max(256, 8 * max(neg_scores.size, pos_scores.size))))
    idx_n = rng.randint(0, neg_scores.size, size=n_pairs)
    idx_p = rng.randint(0, pos_scores.size, size=n_pairs)
    diffs = (pos_scores[idx_p] - neg_scores[idx_n]).astype(np.float32).reshape(-1, 1)
    X = np.concatenate([diffs, -diffs], axis=0)
    y = np.concatenate([
        np.ones((n_pairs,), dtype=np.int64),
        np.zeros((n_pairs,), dtype=np.int64),
    ])
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs")
    clf.fit(Xs, y)
    raw_coef = float(np.abs(clf.coef_[0, 0]) / max(float(scaler.scale_[0]), 1e-6))
    tau = float(0.5 * (np.median(neg_scores) + np.median(pos_scores)))
    temp = float(max(1.0 / max(raw_coef, 1e-3), 1e-3))
    return dict(kind="rank", tau=tau, temp=temp, scaler=scaler, clf=clf, raw_coef=raw_coef)


def fit_hardneg_score_calibrator(neg_scores, pos_scores, q=HARDNEG_Q, dup=HARDNEG_DUP):
    neg_scores = np.asarray(neg_scores, dtype=np.float32).reshape(-1)
    pos_scores = np.asarray(pos_scores, dtype=np.float32).reshape(-1)
    if neg_scores.size == 0:
        return fit_binary_score_calibrator(neg_scores, pos_scores)
    thr = float(np.quantile(neg_scores, q))
    hard = neg_scores[neg_scores >= thr]
    if hard.size > 0 and int(dup) > 0:
        neg_aug = np.concatenate([neg_scores] + [hard for _ in range(int(dup))], axis=0)
    else:
        neg_aug = neg_scores
    return dict(hard_thr=thr, hard_dup=int(dup), base=fit_binary_score_calibrator(neg_aug, pos_scores), kind="hardneg")


def predict_binary_score_calibrator(scores, bundle):
    scores = np.asarray(scores, dtype=np.float32).reshape(-1, 1)
    if bundle["kind"] == "threshold":
        tau = float(bundle["tau"])
        temp = max(float(bundle["temp"]), 1e-6)
        return sigmoid_np((scores[:, 0] - tau) / temp).astype(np.float32)
    if bundle["kind"] == "rank":
        tau = float(bundle["tau"])
        temp = max(float(bundle["temp"]), 1e-6)
        return sigmoid_np((scores[:, 0] - tau) / temp).astype(np.float32)
    if bundle["kind"] == "hardneg":
        return predict_binary_score_calibrator(scores[:, 0], bundle["base"])
    Xs = bundle["scaler"].transform(scores)
    return bundle["clf"].predict_proba(Xs)[:, 1].astype(np.float32)


def fit_classwise_score_calibrators(neg_scores, y_neg, pos_scores, y_pos, K, min_count=24, fit_fn=fit_binary_score_calibrator):
    neg_scores = np.asarray(neg_scores, dtype=np.float32)
    pos_scores = np.asarray(pos_scores, dtype=np.float32)
    y_neg = np.asarray(y_neg, dtype=np.int64)
    y_pos = np.asarray(y_pos, dtype=np.int64)

    global_bundle = fit_fn(neg_scores, pos_scores)
    bundles = []
    for k in range(K):
        neg_k = neg_scores[y_neg == k]
        pos_k = pos_scores[y_pos == k]
        if neg_k.size >= max(int(min_count), 8) and pos_k.size >= max(int(min_count), 8):
            bundles.append(fit_fn(neg_k, pos_k))
        else:
            bundles.append(global_bundle)
    return dict(global_bundle=global_bundle, class_bundles=bundles)


def predict_classwise_score_calibrators(scores, k_hat, bundle):
    scores = np.asarray(scores, dtype=np.float32)
    k_hat = np.asarray(k_hat, dtype=np.int64)
    out = np.zeros((len(scores),), dtype=np.float32)
    for k in np.unique(k_hat):
        idx = np.where(k_hat == k)[0]
        kk = int(k)
        if kk < 0 or kk >= len(bundle["class_bundles"]):
            out[idx] = predict_binary_score_calibrator(scores[idx], bundle["global_bundle"])
        else:
            out[idx] = predict_binary_score_calibrator(scores[idx], bundle["class_bundles"][kk])
    return out.astype(np.float32)


def prepare_source_synthetic_drift_scores(X_cal, y_cal_true, K, rng, models_kr, fb, source_rx_ids,
                                          dom_mu, dom_std, dom_global_mu, dom_global_std,
                                          augment_fn=synthetic_drift_augment_batch,
                                          copies=SYNTH_DRIFT_COPIES):
    X_syn = augment_fn(X_cal, rng, copies=copies)
    y_syn = np.repeat(np.asarray(y_cal_true, dtype=np.int64), int(max(1, copies)))
    D_syn = compute_domain_feats_batch(X_syn, d_dim=D_DIM)
    Sdom_allk_syn = sdom_V1_mixNLL_allk(D_syn, K, models_kr, fb, source_rx_ids)
    Sdrift_allk_syn = normalize_dom_matrix_by_class(Sdom_allk_syn, dom_mu, dom_std, dom_global_mu, dom_global_std)
    Sdrift_syn_true = gather_hard_class_scores(Sdrift_allk_syn, y_syn)
    return Sdrift_syn_true.astype(np.float32), y_syn.astype(np.int64)

def build_variant_router_config(variant_name, Sid_cal, Sdrift_router_cal, Sdrift_true_cal, y_true_cal,
                                synth_bundle_dict, K):
    spec = dict(V71_VARIANT_SPECS[variant_name])
    cfg = dict(
        name=variant_name,
        sequential_unknown_gate=False,  # keep soft tri-state for all V7.1 variants
        classwise_drift=bool(spec["classwise"]),
        attenuation=bool(spec["atten"]),
        synthopt=bool(spec["synthopt"]),
        rank=bool(spec["rank"]),
        hardneg=bool(spec["hardneg"]),
        tau_id=float(np.quantile(np.asarray(Sid_cal, dtype=np.float32), FRR_TARGET)),
        temp_id=float(robust_scale(Sid_cal)),
    )

    # Drift calibrator choice
    if cfg["classwise_drift"]:
        cfg["drift_mode"] = "classwise_synth_calibrator"
        cfg["drift_calibrator"] = fit_classwise_score_calibrators(
            Sdrift_true_cal,
            y_true_cal,
            synth_bundle_dict["base"][0],
            synth_bundle_dict["base"][1],
            K,
            min_count=SYNTH_MIN_CLASS_CAL,
            fit_fn=fit_binary_score_calibrator,
        )
    elif cfg["rank"]:
        cfg["drift_mode"] = "global_rank_calibrator"
        cfg["drift_calibrator"] = fit_pairwise_rank_calibrator(
            Sdrift_true_cal,
            synth_bundle_dict["base"][0],
            max_pairs=RANK_MAX_PAIRS,
        )
    elif cfg["hardneg"]:
        cfg["drift_mode"] = "global_hardneg_calibrator"
        cfg["drift_calibrator"] = fit_hardneg_score_calibrator(
            Sdrift_true_cal,
            synth_bundle_dict["base"][0],
            q=HARDNEG_Q,
            dup=HARDNEG_DUP,
        )
    elif cfg["synthopt"]:
        cfg["drift_mode"] = "global_synthopt_calibrator"
        cfg["drift_calibrator"] = fit_binary_score_calibrator(
            Sdrift_true_cal,
            synth_bundle_dict["opt"][0],
        )
    else:
        cfg["drift_mode"] = "global_synth_calibrator"
        cfg["drift_calibrator"] = fit_binary_score_calibrator(
            Sdrift_true_cal,
            synth_bundle_dict["base"][0],
        )

    if cfg["attenuation"]:
        cfg["atten_alpha"] = float(ATTEN_ALPHA)
        cfg["atten_sid_tau"] = float(np.quantile(np.asarray(Sid_cal, dtype=np.float32), ATTEN_SID_Q))
        cfg["atten_sid_temp"] = float(robust_scale(np.asarray(Sid_cal, dtype=np.float32)))
        cfg["atten_driftref_tau"] = float(np.quantile(np.asarray(Sdrift_router_cal, dtype=np.float32), ATTEN_DRIFTREF_Q))
        cfg["atten_driftref_temp"] = float(robust_scale(np.asarray(Sdrift_router_cal, dtype=np.float32)))
    return cfg


def predict_variant_router(Sid, Sdrift_router, Sdrift_hard, k_hat, cfg):
    Sid = np.asarray(Sid, dtype=np.float32)
    Sdrift_router = np.asarray(Sdrift_router, dtype=np.float32)
    Sdrift_hard = np.asarray(Sdrift_hard, dtype=np.float32)
    k_hat = np.asarray(k_hat, dtype=np.int64)

    tau_id = float(cfg["tau_id"])
    temp_id = max(float(cfg["temp_id"]), 1e-6)
    p_unknown = sigmoid_np((tau_id - Sid) / temp_id).astype(np.float32)
    p_known = 1.0 - p_unknown

    if cfg["drift_mode"] == "classwise_synth_calibrator":
        p_drift_cond = predict_classwise_score_calibrators(Sdrift_hard, k_hat, cfg["drift_calibrator"])
    else:
        score = Sdrift_router
        p_drift_cond = predict_binary_score_calibrator(score, cfg["drift_calibrator"])

    # Stable-protection attenuation: damp drift probability for highly confident stable-known source-like samples.
    if cfg.get("attenuation", False):
        sid_hi = sigmoid_np((Sid - float(cfg["atten_sid_tau"])) / max(float(cfg["atten_sid_temp"]), 1e-6)).astype(np.float32)
        drift_borderline = 1.0 - sigmoid_np((Sdrift_router - float(cfg["atten_driftref_tau"])) / max(float(cfg["atten_driftref_temp"]), 1e-6)).astype(np.float32)
        atten = 1.0 - float(cfg["atten_alpha"]) * sid_hi * drift_borderline
        atten = np.clip(atten, 0.20, 1.00).astype(np.float32)
        p_drift_cond = np.clip(p_drift_cond * atten, 0.0, 1.0).astype(np.float32)

    probs = np.stack([
        p_known * (1.0 - p_drift_cond),
        p_known * p_drift_cond,
        p_unknown,
    ], axis=1).astype(np.float32)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)
    pred = np.argmax(probs, axis=1).astype(np.int64)
    aux = dict(
        p_unknown=p_unknown.astype(np.float32),
        p_drift_cond=p_drift_cond.astype(np.float32),
    )
    return pred, probs, aux


def summarize_variant_cfg(cfg):
    out = {}
    for k, v in cfg.items():
        if isinstance(v, (bool, int, float, str)):
            out[k] = v
        elif isinstance(v, np.ndarray):
            out[k] = dict(shape=list(v.shape), mean=float(np.mean(v)), std=float(np.std(v)))
        elif isinstance(v, dict):
            out[k] = dict(keys=list(v.keys()))
    return out



# =========================
# Main loop
# =========================
def run_one_split(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id, KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx):
    protocol_dir = os.path.join(RUN_DIR, protocol_tag)
    split_dir = os.path.join(protocol_dir, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "protocol_tag": protocol_tag,
                "rx_protocol": rx_protocol_tag,
                "tx_protocol": tx_protocol_tag,
                "KNOWN_TX": KNOWN_TX,
                "UNKNOWN_TX": UNKNOWN_TX,
                "SOURCE_RXS": source_rxs,
                "DRIFT_RX": drift_rx,
            },
            f,
            indent=2,
        )

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX, source_rxs)
    K = len(KNOWN_TX)

    X_drRX, D_drRX = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, drift_rx, TEST_DATES_RX)
    X_drDAY, D_drDAY = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, source_rxs, TEST_DATES_DAY)

    X_u_in, D_u_in = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, source_rxs, TEST_DATES_RX)
    X_u_drRX, D_u_drRX = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, drift_rx, TEST_DATES_RX)
    X_u_drDAY, D_u_drDAY = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, source_rxs, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000 * split_id + len(source_rxs))
    rows = []

    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rng = np.random.RandomState(SEED + 1000 * split_id + 100 * len(source_rxs) + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(ArrayDataset(X_src[idx_val], y_src[idx_val]), batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val = -1.0
        best_state = None
        patience = 0
        hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        for ep in range(1, EPOCHS + 1):
            tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=opt)
            va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
            hist["train_loss"].append(tr_loss)
            hist["train_acc"].append(tr_acc)
            hist["val_loss"].append(va_loss)
            hist["val_acc"].append(va_acc)
            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
        with open(os.path.join(fold_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(hist, f, indent=2)
        model.load_state_dict(best_state)

        X_A_all = X_src[idx_te]
        y_A_true = y_src[idx_te]
        D_A = D_src[idx_te]

        logits_A, Z_A = infer_logits_embed(model, X_A_all, batch=512)
        acc = float((np.argmax(logits_A, 1) == y_A_true).mean())

        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        mu_z, var_z = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)
        P = prototypes(Z_tr, y_src[idx_train], K)

        logits_B, Z_B = infer_logits_embed(model, X_drRX, batch=512)
        logits_C, Z_C = infer_logits_embed(model, X_drDAY, batch=512)
        logits_D, Z_D = infer_logits_embed(model, X_u_in, batch=512)
        logits_E, Z_E = infer_logits_embed(model, X_u_drRX, batch=512)
        logits_F, Z_F = infer_logits_embed(model, X_u_drDAY, batch=512)

        SidA_emb = sid_EmbMaha(Z_A, mu_z, var_z)
        SidB_emb = sid_EmbMaha(Z_B, mu_z, var_z)
        SidC_emb = sid_EmbMaha(Z_C, mu_z, var_z)
        SidD_emb = sid_EmbMaha(Z_D, mu_z, var_z)
        SidE_emb = sid_EmbMaha(Z_E, mu_z, var_z)
        SidF_emb = sid_EmbMaha(Z_F, mu_z, var_z)

        SidA_en = sid_Energy(logits_A)
        SidB_en = sid_Energy(logits_B)
        SidC_en = sid_Energy(logits_C)
        SidD_en = sid_Energy(logits_D)
        SidE_en = sid_Energy(logits_E)
        SidF_en = sid_Energy(logits_F)

        zA_emb = zscore(SidA_emb, SidA_emb)
        zB_emb = zscore(SidB_emb, SidA_emb)
        zC_emb = zscore(SidC_emb, SidA_emb)
        zD_emb = zscore(SidD_emb, SidA_emb)
        zE_emb = zscore(SidE_emb, SidA_emb)
        zF_emb = zscore(SidF_emb, SidA_emb)

        zA_en = zscore(SidA_en, SidA_en)
        zB_en = zscore(SidB_en, SidA_en)
        zC_en = zscore(SidC_en, SidA_en)
        zD_en = zscore(SidD_en, SidA_en)
        zE_en = zscore(SidE_en, SidA_en)
        zF_en = zscore(SidF_en, SidA_en)

        Sid_A = (zA_emb + FUSION_LAM * zA_en).astype(np.float32)
        Sid_B = (zB_emb + FUSION_LAM * zB_en).astype(np.float32)
        Sid_C = (zC_emb + FUSION_LAM * zC_en).astype(np.float32)
        Sid_D = (zD_emb + FUSION_LAM * zD_en).astype(np.float32)
        Sid_E = (zE_emb + FUSION_LAM * zE_en).astype(np.float32)
        Sid_F = (zF_emb + FUSION_LAM * zF_en).astype(np.float32)

        Sidtr_emb = sid_EmbMaha(Z_tr, mu_z, var_z)
        Sidtr_en = sid_Energy(logits_tr)
        ztr_emb = zscore(Sidtr_emb, Sidtr_emb)
        ztr_en = zscore(Sidtr_en, Sidtr_en)
        Sid_tr = (ztr_emb + FUSION_LAM * ztr_en).astype(np.float32)

        CosA = cosine_to_proto(Z_A, P)
        CosB = cosine_to_proto(Z_B, P)
        CosC = cosine_to_proto(Z_C, P)
        CosD = cosine_to_proto(Z_D, P)
        CosE = cosine_to_proto(Z_E, P)
        CosF = cosine_to_proto(Z_F, P)

        kA = np.argmax(CosA, 1).astype(np.int64)
        kB = np.argmax(CosB, 1).astype(np.int64)
        kC = np.argmax(CosC, 1).astype(np.int64)
        kD = np.argmax(CosD, 1).astype(np.int64)
        kE = np.argmax(CosE, 1).astype(np.int64)
        kF = np.argmax(CosF, 1).astype(np.int64)

        source_rx_ids = sorted({RX_USE.index(r) for r in source_rxs})
        models_kr, fb = fit_device_rx_models(
            D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20
        )

        Sdom_tr_true = sdom_V1_mixNLL(D_src[idx_train], y_src[idx_train], models_kr, fb, source_rx_ids)
        dom_keep_mask, dom_sid_thr, dom_keep_counts = select_high_sid_subset_by_class(
            Sid_tr, y_src[idx_train], K, q=DOM_STATS_SID_Q, min_keep_per_class=DOM_STATS_MIN_KEEP_PER_CLASS
        )
        if int(dom_keep_mask.sum()) < max(K * 8, 32):
            dom_keep_mask = np.ones_like(dom_keep_mask, dtype=bool)
            dom_sid_thr = np.full((K,), float(np.min(Sid_tr)), dtype=np.float32)
            dom_keep_counts = np.array([(y_src[idx_train] == k).sum() for k in range(K)], dtype=np.int64)

        dom_mu, dom_std, dom_global_mu, dom_global_std = fit_classwise_dom_stats(
            Sdom_tr_true[dom_keep_mask],
            y_src[idx_train][dom_keep_mask],
            K,
            min_std=1e-3,
        )
        dom_keep_frac = float(np.mean(dom_keep_mask.astype(np.float32)))
        dom_keep_mean = float(np.mean(dom_keep_counts.astype(np.float32)))
        dom_keep_min = int(np.min(dom_keep_counts)) if dom_keep_counts.size > 0 else 0

        Sdom_allk_A = sdom_V1_mixNLL_allk(D_A, K, models_kr, fb, source_rx_ids)
        Sdom_allk_B = sdom_V1_mixNLL_allk(D_drRX, K, models_kr, fb, source_rx_ids)
        Sdom_allk_C = sdom_V1_mixNLL_allk(D_drDAY, K, models_kr, fb, source_rx_ids)
        Sdom_allk_D = sdom_V1_mixNLL_allk(D_u_in, K, models_kr, fb, source_rx_ids)
        Sdom_allk_E = sdom_V1_mixNLL_allk(D_u_drRX, K, models_kr, fb, source_rx_ids)
        Sdom_allk_F = sdom_V1_mixNLL_allk(D_u_drDAY, K, models_kr, fb, source_rx_ids)

        Sdrift_allk_A = normalize_dom_matrix_by_class(Sdom_allk_A, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_B = normalize_dom_matrix_by_class(Sdom_allk_B, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_C = normalize_dom_matrix_by_class(Sdom_allk_C, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_D = normalize_dom_matrix_by_class(Sdom_allk_D, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_E = normalize_dom_matrix_by_class(Sdom_allk_E, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_F = normalize_dom_matrix_by_class(Sdom_allk_F, dom_mu, dom_std, dom_global_mu, dom_global_std)

        Sdom_hard_A = gather_hard_class_scores(Sdom_allk_A, kA)
        Sdom_hard_B = gather_hard_class_scores(Sdom_allk_B, kB)
        Sdom_hard_C = gather_hard_class_scores(Sdom_allk_C, kC)
        Sdom_hard_D = gather_hard_class_scores(Sdom_allk_D, kD)
        Sdom_hard_E = gather_hard_class_scores(Sdom_allk_E, kE)
        Sdom_hard_F = gather_hard_class_scores(Sdom_allk_F, kF)

        Sdrift_hard_A = gather_hard_class_scores(Sdrift_allk_A, kA)
        Sdrift_hard_B = gather_hard_class_scores(Sdrift_allk_B, kB)
        Sdrift_hard_C = gather_hard_class_scores(Sdrift_allk_C, kC)
        Sdrift_hard_D = gather_hard_class_scores(Sdrift_allk_D, kD)
        Sdrift_hard_E = gather_hard_class_scores(Sdrift_allk_E, kE)
        Sdrift_hard_F = gather_hard_class_scores(Sdrift_allk_F, kF)

        Sdom_soft_A, _, _ = aggregate_top2_scores(Sdom_allk_A, CosA, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_B, _, _ = aggregate_top2_scores(Sdom_allk_B, CosB, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_C, _, _ = aggregate_top2_scores(Sdom_allk_C, CosC, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_D, _, _ = aggregate_top2_scores(Sdom_allk_D, CosD, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_E, _, _ = aggregate_top2_scores(Sdom_allk_E, CosE, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_F, _, _ = aggregate_top2_scores(Sdom_allk_F, CosF, temp=ROUTER_TOP2_TEMP)

        Sdrift_soft_A, _, _ = aggregate_top2_scores(Sdrift_allk_A, CosA, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_B, _, _ = aggregate_top2_scores(Sdrift_allk_B, CosB, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_C, _, _ = aggregate_top2_scores(Sdrift_allk_C, CosC, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_D, _, _ = aggregate_top2_scores(Sdrift_allk_D, CosD, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_E, _, _ = aggregate_top2_scores(Sdrift_allk_E, CosE, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_F, _, _ = aggregate_top2_scores(Sdrift_allk_F, CosF, temp=ROUTER_TOP2_TEMP)

        if ROUTER_FEATURE_MODE == "hardk":
            JR_Sdom_A, JR_Sdom_B, JR_Sdom_C = Sdom_hard_A, Sdom_hard_B, Sdom_hard_C
            JR_Sdom_D, JR_Sdom_E, JR_Sdom_F = Sdom_hard_D, Sdom_hard_E, Sdom_hard_F
            JR_Sdr_A, JR_Sdr_B, JR_Sdr_C = Sdrift_hard_A, Sdrift_hard_B, Sdrift_hard_C
            JR_Sdr_D, JR_Sdr_E, JR_Sdr_F = Sdrift_hard_D, Sdrift_hard_E, Sdrift_hard_F
        else:
            JR_Sdom_A, JR_Sdom_B, JR_Sdom_C = Sdom_soft_A, Sdom_soft_B, Sdom_soft_C
            JR_Sdom_D, JR_Sdom_E, JR_Sdom_F = Sdom_soft_D, Sdom_soft_E, Sdom_soft_F
            JR_Sdr_A, JR_Sdr_B, JR_Sdr_C = Sdrift_soft_A, Sdrift_soft_B, Sdrift_soft_C
            JR_Sdr_D, JR_Sdr_E, JR_Sdr_F = Sdrift_soft_D, Sdrift_soft_E, Sdrift_soft_F

        rng_router = np.random.RandomState(SEED + 1000 * split_id + 100 * len(source_rxs) + 10 * fold)
        cal_A, ev_A = split_cal_eval_indices(len(Sid_A), ROUTER_CAL_FRAC, rng_router, ROUTER_MIN_CAL_PER_POOL)
        ev_B = np.arange(len(Sid_B), dtype=np.int64)
        ev_C = np.arange(len(Sid_C), dtype=np.int64)
        ev_D = np.arange(len(Sid_D), dtype=np.int64)
        ev_E = np.arange(len(Sid_E), dtype=np.int64)
        ev_F = np.arange(len(Sid_F), dtype=np.int64)

        tau_id, tau_dom = calibrate_thresholds(Sid_A[cal_A], Sdom_hard_A[cal_A], FRR_TARGET, FALSE_DRIFT_TARGET)

        pred_A_base = gate_predict(Sid_A[ev_A], Sdom_hard_A[ev_A], tau_id, tau_dom)
        pred_B_base = gate_predict(Sid_B[ev_B], Sdom_hard_B[ev_B], tau_id, tau_dom)
        pred_C_base = gate_predict(Sid_C[ev_C], Sdom_hard_C[ev_C], tau_id, tau_dom)
        pred_D_base = gate_predict(Sid_D[ev_D], Sdom_hard_D[ev_D], tau_id, tau_dom)
        pred_E_base = gate_predict(Sid_E[ev_E], Sdom_hard_E[ev_E], tau_id, tau_dom)
        pred_F_base = gate_predict(Sid_F[ev_F], Sdom_hard_F[ev_F], tau_id, tau_dom)

        base_metrics, cm_base, gt_eval, pred_eval_base = compute_state_metrics(
            pred_A_base, pred_B_base, pred_C_base, pred_D_base, pred_E_base, pred_F_base
        )
        # Source-only stable calibration scores for V7.1 variants
        Sdrift_true_A_cal = gather_hard_class_scores(Sdrift_allk_A[cal_A], y_A_true[cal_A])

        variant_outputs = {}
        variant_cfgs = {}

        # Source-only synthetic drift positives for the V7.1 variants
        rng_syn = np.random.RandomState(SEED + 1000 * split_id + 100 * len(source_rxs) + 1000 * fold + 7)
        synth_bundle_dict = {}
        synth_bundle_dict["base"] = prepare_source_synthetic_drift_scores(
            X_A_all[cal_A], y_A_true[cal_A], K, rng_syn, models_kr, fb, source_rx_ids, dom_mu, dom_std, dom_global_mu, dom_global_std,
            augment_fn=synthetic_drift_augment_batch, copies=SYNTH_DRIFT_COPIES
        )
        rng_syn_opt = np.random.RandomState(SEED + 1000 * split_id + 100 * len(source_rxs) + 1000 * fold + 19)
        synth_bundle_dict["opt"] = prepare_source_synthetic_drift_scores(
            X_A_all[cal_A], y_A_true[cal_A], K, rng_syn_opt, models_kr, fb, source_rx_ids, dom_mu, dom_std, dom_global_mu, dom_global_std,
            augment_fn=synthetic_drift_augment_batch_v2, copies=SYNTHOPT_DRIFT_COPIES
        )

        for vname in V71_VARIANT_ORDER:
            cfg = build_variant_router_config(
                vname,
                Sid_A[cal_A],
                JR_Sdr_A[cal_A],
                Sdrift_true_A_cal,
                y_A_true[cal_A],
                synth_bundle_dict,
                K,
            )
            variant_cfgs[vname] = cfg

            pred_A, probs_A, aux_A = predict_variant_router(Sid_A[ev_A], JR_Sdr_A[ev_A], Sdrift_hard_A[ev_A], kA[ev_A], cfg)
            pred_B, probs_B, aux_B = predict_variant_router(Sid_B[ev_B], JR_Sdr_B[ev_B], Sdrift_hard_B[ev_B], kB[ev_B], cfg)
            pred_C, probs_C, aux_C = predict_variant_router(Sid_C[ev_C], JR_Sdr_C[ev_C], Sdrift_hard_C[ev_C], kC[ev_C], cfg)
            pred_D, probs_D, aux_D = predict_variant_router(Sid_D[ev_D], JR_Sdr_D[ev_D], Sdrift_hard_D[ev_D], kD[ev_D], cfg)
            pred_E, probs_E, aux_E = predict_variant_router(Sid_E[ev_E], JR_Sdr_E[ev_E], Sdrift_hard_E[ev_E], kE[ev_E], cfg)
            pred_F, probs_F, aux_F = predict_variant_router(Sid_F[ev_F], JR_Sdr_F[ev_F], Sdrift_hard_F[ev_F], kF[ev_F], cfg)

            m_v, cm_v, _, _ = compute_state_metrics(pred_A, pred_B, pred_C, pred_D, pred_E, pred_F)

            auc_unknown_routerprob = roc_auc(
                np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_D),), dtype=np.int64)]),
                np.concatenate([probs_A[:, 2], probs_D[:, 2]])
            )
            auc_drift_rx_router = roc_auc(
                np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_B),), dtype=np.int64)]),
                np.concatenate([probs_A[:, 1], probs_B[:, 1]])
            )
            auc_drift_day_router = roc_auc(
                np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_C),), dtype=np.int64)]),
                np.concatenate([probs_A[:, 1], probs_C[:, 1]])
            )

            variant_outputs[vname] = dict(
                metrics=m_v,
                cm=cm_v,
                probs_A=probs_A,
                probs_B=probs_B,
                probs_C=probs_C,
                probs_D=probs_D,
                probs_E=probs_E,
                probs_F=probs_F,
                aux_A=aux_A,
                aux_B=aux_B,
                aux_C=aux_C,
                aux_D=aux_D,
                aux_E=aux_E,
                aux_F=aux_F,
                auc_unknown_routerprob=float(auc_unknown_routerprob),
                auc_drift_rx_router=float(auc_drift_rx_router),
                auc_drift_day_router=float(auc_drift_day_router),
                cfg_summary=summarize_variant_cfg(cfg),
            )

        auc_unknown_sid = roc_auc(
            np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_D),), dtype=np.int64)]),
            np.concatenate([-Sid_A[ev_A], -Sid_D[ev_D]])
        )
        auc_drift_rx_raw_hard = roc_auc(
            np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_B),), dtype=np.int64)]),
            np.concatenate([Sdom_hard_A[ev_A], Sdom_hard_B[ev_B]])
        )
        auc_drift_day_raw_hard = roc_auc(
            np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_C),), dtype=np.int64)]),
            np.concatenate([Sdom_hard_A[ev_A], Sdom_hard_C[ev_C]])
        )

        Sid_all_eval = np.concatenate([Sid_A[ev_A], Sid_B[ev_B], Sid_C[ev_C], Sid_D[ev_D], Sid_E[ev_E], Sid_F[ev_F]])
        Sdom_all_eval = np.concatenate([Sdom_hard_A[ev_A], Sdom_hard_B[ev_B], Sdom_hard_C[ev_C], Sdom_hard_D[ev_D], Sdom_hard_E[ev_E], Sdom_hard_F[ev_F]])
        Sdrift_all_eval = np.concatenate([JR_Sdr_A[ev_A], JR_Sdr_B[ev_B], JR_Sdr_C[ev_C], JR_Sdr_D[ev_D], JR_Sdr_E[ev_E], JR_Sdr_F[ev_F]])
        gt_all_eval = gt_eval.copy()

        plot_gate_scatter(
            Sid_all_eval, Sdom_all_eval, gt_all_eval, tau_id, tau_dom,
            os.path.join(fold_dir, "baseline_gate_scatter_eval.png")
        )
        plot_joint_scatter(
            Sid_all_eval, Sdrift_all_eval, gt_all_eval,
            os.path.join(fold_dir, "v7_1_joint_scatter_eval.png")
        )
        plot_confmat(cm_base, os.path.join(fold_dir, "baseline_confmat_eval.png"), "Baseline hard gate (eval)")
        for vname in V71_VARIANT_ORDER:
            plot_confmat(
                variant_outputs[vname]["cm"],
                os.path.join(fold_dir, f"{vname}_confmat_eval.png"),
                f"{vname} (eval)"
            )

        metrics = dict(
            protocol_tag=protocol_tag,
            rx_protocol=rx_protocol_tag,
            tx_protocol=tx_protocol_tag,
            split=split_id,
            fold=fold,
            acc=acc,
            router_feature_mode=ROUTER_FEATURE_MODE,
            router_cal_frac=ROUTER_CAL_FRAC,
            router_cal_n_stable=int(len(cal_A)),
            dom_stats_sid_q=float(DOM_STATS_SID_Q),
            dom_stats_keep_frac=float(dom_keep_frac),
            dom_stats_keep_mean=float(dom_keep_mean),
            dom_stats_keep_min=int(dom_keep_min),
            dom_stats_keep_counts=dom_keep_counts.tolist(),
            dom_stats_sid_thresholds=dom_sid_thr.tolist(),
            tau_id=float(tau_id),
            tau_dom=float(tau_dom),
            auc_unknown_sid=float(auc_unknown_sid),
            auc_drift_rx_raw_hard=float(auc_drift_rx_raw_hard),
            auc_drift_day_raw_hard=float(auc_drift_day_raw_hard),
            base_confusion_matrix=cm_base.tolist(),
            synth_drift_copies=int(SYNTH_DRIFT_COPIES),
            synth_drift_n_pos_base=int(len(synth_bundle_dict["base"][0])),
            synth_drift_n_pos_opt=int(len(synth_bundle_dict["opt"][0])),
        )
        metrics.update(prefix_keys(base_metrics, "base_"))

        for vname in V71_VARIANT_ORDER:
            vout = variant_outputs[vname]
            metrics.update(prefix_keys(vout["metrics"], f"{vname}_"))
            metrics[f"{vname}_auc_unknown_routerprob"] = float(vout["auc_unknown_routerprob"])
            metrics[f"{vname}_auc_drift_rx_router"] = float(vout["auc_drift_rx_router"])
            metrics[f"{vname}_auc_drift_day_router"] = float(vout["auc_drift_day_router"])
            metrics[f"{vname}_confusion_matrix"] = vout["cm"].tolist()
            for k_cfg, v_cfg in vout["cfg_summary"].items():
                if isinstance(v_cfg, (bool, int, float, str)):
                    metrics[f"{vname}_cfg_{k_cfg}"] = v_cfg

            for key in [
                "stable_acc_gate",
                "drift_acc_all",
                "unknown_reject_all",
                "FRR_stable",
                "FAR_unknown_accept",
                "false_unknown_drift_all",
                "false_drift_unknown",
            ]:
                metrics[f"delta_{vname}_{key}"] = float(metrics[f"{vname}_{key}"] - metrics[f"base_{key}"])

        with open(os.path.join(fold_dir, "metrics_v7_1_ablation.json"), "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)

        rows.append(metrics)

        msg = [
            f"[{protocol_tag} split {split_id} fold {fold}]",
            f"base(d={metrics['base_drift_acc_all']:.3f}, u={metrics['base_unknown_reject_all']:.3f})",
        ]
        for vname in V71_VARIANT_ORDER:
            msg.append(
                f"{vname}(d={metrics[f'{vname}_drift_acc_all']:.3f}, "
                f"u={metrics[f'{vname}_unknown_reject_all']:.3f}, "
                f"Δd={metrics[f'delta_{vname}_drift_acc_all']:+.3f}, "
                f"Δu={metrics[f'delta_{vname}_unknown_reject_all']:+.3f})"
            )
        print(" | ".join(msg))

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows = []
for ps in PROTOCOL_SPECS:
    protocol_tag = ps["protocol_tag"]
    print("\n==============================")
    print("Protocol:", protocol_tag)
    print("SOURCE_RXS:", ps["source_rxs"])
    print("DRIFT_RX:", ps["drift_rx"])
    print("==============================")
    for split_pos, tx_split in enumerate(ps["tx_splits"], start=1):
        # Compatibility across protocol spec formats:
        # new format: {'split_id','known','unknown'}
        # older format: {'KNOWN_TX','UNKNOWN_TX','tx_protocol_tag'}
        # tuple format: (KNOWN_TX, UNKNOWN_TX)
        if isinstance(tx_split, dict):
            KNOWN_TX = tx_split.get("KNOWN_TX", tx_split.get("known"))
            UNKNOWN_TX = tx_split.get("UNKNOWN_TX", tx_split.get("unknown"))
            tx_protocol_tag = tx_split.get("tx_protocol_tag", tx_split.get("tx_protocol", ps.get("tx_protocol_tag", ps.get("tx_protocol"))))
            if KNOWN_TX is None or UNKNOWN_TX is None:
                raise KeyError(f"Unrecognized tx_split dict keys: {list(tx_split.keys())}")
        else:
            KNOWN_TX, UNKNOWN_TX = tx_split
            tx_protocol_tag = ps.get("tx_protocol_tag", ps.get("tx_protocol"))

        rows = run_one_split(
            protocol_tag=protocol_tag,
            rx_protocol_tag=ps.get("rx_protocol_tag", ps.get("rx_protocol")),
            tx_protocol_tag=tx_protocol_tag,
            split_id=split_pos,
            KNOWN_TX=KNOWN_TX,
            UNKNOWN_TX=UNKNOWN_TX,
            source_rxs=ps["source_rxs"],
            drift_rx=ps["drift_rx"],
        )
        all_rows.extend(rows)

with open(os.path.join(RUN_DIR, "metrics_all_protocols.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)

print("\nDone. RUN_DIR =", RUN_DIR)



# =========================
# Summary
# =========================
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

BASE_KEYS = [
    "base_stable_acc_gate",
    "base_drift_acc_all",
    "base_unknown_reject_all",
    "base_FRR_stable",
    "base_FAR_unknown_accept",
    "base_false_unknown_drift_all",
]

VARIANT_CORE_KEYS = [
    "stable_acc_gate",
    "drift_acc_all",
    "unknown_reject_all",
    "FRR_stable",
    "FAR_unknown_accept",
    "false_unknown_drift_all",
    "auc_unknown_routerprob",
    "auc_drift_rx_router",
    "auc_drift_day_router",
]

COMMON_KEYS = [
    "acc",
    "auc_unknown_sid",
    "auc_drift_rx_raw_hard",
    "auc_drift_day_raw_hard",
    "tau_id",
    "tau_dom",
    "dom_stats_keep_frac",
    "dom_stats_keep_mean",
    "dom_stats_keep_min",
    "synth_drift_n_pos_base",
    "synth_drift_n_pos_opt",
]

SUMMARY_KEYS = COMMON_KEYS + BASE_KEYS
for vname in V71_VARIANT_ORDER:
    SUMMARY_KEYS.extend([f"{vname}_{k}" for k in VARIANT_CORE_KEYS])
    SUMMARY_KEYS.extend([
        f"delta_{vname}_stable_acc_gate",
        f"delta_{vname}_drift_acc_all",
        f"delta_{vname}_unknown_reject_all",
        f"delta_{vname}_FRR_stable",
        f"delta_{vname}_FAR_unknown_accept",
        f"delta_{vname}_false_unknown_drift_all",
    ])

summary_overall = {}
for key in SUMMARY_KEYS:
    m, s = mean_std([r[key] for r in all_rows])
    summary_overall[key] = {"mean": m, "std": s}

cm_base_sum = np.zeros((3, 3), dtype=np.int64)
variant_cm_sums = {vname: np.zeros((3, 3), dtype=np.int64) for vname in V71_VARIANT_ORDER}
for r in all_rows:
    cm_base_sum += np.array(r["base_confusion_matrix"], dtype=np.int64)
    for vname in V71_VARIANT_ORDER:
        variant_cm_sums[vname] += np.array(r[f"{vname}_confusion_matrix"], dtype=np.int64)

summary_overall["base_confusion_matrix_sum"] = cm_base_sum.tolist()
for vname in V71_VARIANT_ORDER:
    summary_overall[f"{vname}_confusion_matrix_sum"] = variant_cm_sums[vname].tolist()

summary_by_protocol = {}
protocol_tags = sorted(set(r["protocol_tag"] for r in all_rows))
for ptag in protocol_tags:
    rows_p = [r for r in all_rows if r["protocol_tag"] == ptag]
    block = {"n_rows": len(rows_p)}
    for key in SUMMARY_KEYS:
        m, s = mean_std([r[key] for r in rows_p])
        block[key] = {"mean": m, "std": s}

    cm_b = np.zeros((3, 3), dtype=np.int64)
    cm_vars = {vname: np.zeros((3, 3), dtype=np.int64) for vname in V71_VARIANT_ORDER}
    for r in rows_p:
        cm_b += np.array(r["base_confusion_matrix"], dtype=np.int64)
        for vname in V71_VARIANT_ORDER:
            cm_vars[vname] += np.array(r[f"{vname}_confusion_matrix"], dtype=np.int64)
    block["base_confusion_matrix_sum"] = cm_b.tolist()
    for vname in V71_VARIANT_ORDER:
        block[f"{vname}_confusion_matrix_sum"] = cm_vars[vname].tolist()

    # Simple ranking by drift_acc_all first, then unknown_reject_all, then FRR_stable (lower better)
    ranking = []
    for vname in V71_VARIANT_ORDER:
        ranking.append(dict(
            variant=vname,
            drift_acc_all=block[f"{vname}_drift_acc_all"]["mean"],
            unknown_reject_all=block[f"{vname}_unknown_reject_all"]["mean"],
            FRR_stable=block[f"{vname}_FRR_stable"]["mean"],
            delta_drift_acc_all=block[f"delta_{vname}_drift_acc_all"]["mean"],
            delta_unknown_reject_all=block[f"delta_{vname}_unknown_reject_all"]["mean"],
            delta_FRR_stable=block[f"delta_{vname}_FRR_stable"]["mean"],
        ))
    ranking = sorted(
        ranking,
        key=lambda z: (-z["drift_acc_all"], -z["unknown_reject_all"], z["FRR_stable"])
    )
    block["variant_ranking"] = ranking
    summary_by_protocol[ptag] = block

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary_overall, f, indent=2)
with open(os.path.join(RUN_DIR, "summary_by_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(summary_by_protocol, f, indent=2)

print("=== Overall Mean ± Std over protocols × splits × folds ===")
for k in COMMON_KEYS + BASE_KEYS:
    v = summary_overall[k]
    print(f"{k:32s}: {v['mean']:.3f} ± {v['std']:.3f}")

print("\n=== V7.1 variant core metrics (overall) ===")
for vname in V71_VARIANT_ORDER:
    print(f"\n[{vname}]")
    for key in [
        f"{vname}_drift_acc_all",
        f"{vname}_unknown_reject_all",
        f"{vname}_FRR_stable",
        f"{vname}_false_unknown_drift_all",
        f"delta_{vname}_drift_acc_all",
        f"delta_{vname}_unknown_reject_all",
        f"delta_{vname}_FRR_stable",
    ]:
        v = summary_overall[key]
        print(f"  {key:30s}: {v['mean']:.3f} ± {v['std']:.3f}")

print("\nBase confusion matrix SUM (rows=GT, cols=Pred) [Stable,Drift,Unknown]:")
print(cm_base_sum)
for vname in V71_VARIANT_ORDER:
    print(f"\n{vname} confusion matrix SUM (rows=GT, cols=Pred) [Stable,Drift,Unknown]:")
    print(variant_cm_sums[vname])

print("\n=== Per-protocol ranking ===")
for ptag, block in summary_by_protocol.items():
    print(f"\n[{ptag}] n_rows={block['n_rows']}")
    for row in block["variant_ranking"]:
        print(
            f"  {row['variant']:7s} | drift={row['drift_acc_all']:.3f} | "
            f"unkrej={row['unknown_reject_all']:.3f} | FRR={row['FRR_stable']:.3f} | "
            f"Δdrift={row['delta_drift_acc_all']:+.3f} | Δunkrej={row['delta_unknown_reject_all']:+.3f}"
        )
